# **Fine-Tuning BERT for Chunking**

### **Install Required Libraries**

In [1]:
!pip install transformers datasets evaluate seqeval accelerate -U

### **Task 1: Dataset Selection & Task 2: Data Preprocessing**

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

# Task 1: Load WikiANN dataset (English version)
dataset = load_dataset("wikiann", "en")

# Define the model checkpoint
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Identify Label Types (NER/Chunk tags: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC)
label_list = dataset["train"].features["ner_tags"].feature.names
print("Label Categories:", label_list)

# Task 2: Data Preprocessing - Tokenization & Label Alignment
def tokenize_and_align_labels(examples):
    # Tokenize the input, ensuring words are split
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens get mapped to -100 (ignored by PyTorch loss function)
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Subwords get mapped to -100 to avoid duplicate penalization
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply preprocessing to the whole dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("Preprocessing complete. Keys:", tokenized_datasets["train"].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

...

Label Categories: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Preprocessing complete. Keys: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


### **Task 3: Model Setup**

In [3]:
from transformers import AutoModelForTokenClassification

# Create label mappings required for the assignment
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {v: k for k, v in id2label.items()}

# Load the Pre-trained Transformer Model
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded with {model.config.num_labels} labels.")

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded with 7 labels.


### **Task 4: Training & Task 5: Evaluation Setup**

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import os

# 1. Load seqeval metric
metric = evaluate.load("seqeval")

# 2. Define evaluation metrics function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) to compute exact metrics
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

# 3. Define Training Arguments
args = TrainingArguments(
    output_dir="./distilbert-wikiann-chunking",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

# 4. Use data collator to pad inputs and labels dynamically
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# --- THE ULTIMATE NO-TRAINER-EVALUATE LOOP ---
total_epochs = 3
output_base_dir = "./distilbert-wikiann-chunking"

for epoch in range(1, total_epochs + 1):
    print(f"\n Starting Epoch {epoch}/{total_epochs}...")

    # 1. Train for exactly 1 epoch
    trainer.train()

    print(f" Running manual evaluation for Epoch {epoch}...")

    # 2. Get raw predictions instead of using trainer.evaluate()
    # This bypasses the callback and notebook tracking bugs!
    predictions_output = trainer.predict(tokenized_datasets["validation"])

    # 3. Calculate metrics using your already defined compute_metrics function
    metrics = compute_metrics((predictions_output.predictions, predictions_output.label_ids))
    print(f"Epoch {epoch} Results:", metrics)

    # 4. Save your model weights
    epoch_save_path = os.path.join(output_base_dir, f"checkpoint-epoch-{epoch}")
    print(f" Saving model checkpoint to {epoch_save_path}...")
    trainer.save_model(epoch_save_path)

    # 5. Save the tokenizer right next to it manually
    tokenizer.save_pretrained(epoch_save_path)

print("\n All 3 epochs completed, evaluated, and saved successfully!")


 Starting Epoch 1/3...


Step,Training Loss
500,0.010610
1000,0.005874


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Running manual evaluation for Epoch 3...


Epoch 3 Results: {'precision': np.float64(0.7957847350112544), 'recall': np.float64(0.8247561148027711), 'f1': np.float64(0.8100114555489986)}
 Saving model checkpoint to ./distilbert-wikiann-chunking/checkpoint-epoch-3...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 All 3 epochs completed, evaluated, and saved successfully!


In [14]:
from IPython.display import display
import pandas as pd

# 1. Ask the trainer to get predictions for the validation dataset again
print(" Fetching validation results...")
val_predictions = trainer.predict(tokenized_datasets["validation"])

# 2. Use your existing compute_metrics function to get the actual scores
metrics = compute_metrics((val_predictions.predictions, val_predictions.label_ids))

# 3. Put the results in a clean list (pretending this is for epoch 3)
results_data = [{
    "epoch": 3,
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"]
}]

# 4. Create and display the beautiful Pandas DataFrame
df_results = pd.DataFrame(results_data)

print("\n Final Evaluation Summary:")
display(df_results)

 Fetching validation results...



 Final Evaluation Summary:


,epoch,precision,recall,f1
0,3,0.795785,0.824756,0.810011


### **Task 6: Inference**

In [15]:
from transformers import pipeline

# Create a token classification pipeline with our fine-tuned model
# aggregation_strategy="simple" merges Subwords back into whole words
token_classifier = pipeline(
    "token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# Custom sentence from the instructions
custom_text = "John works at Google in California"
predictions = token_classifier(custom_text)

print(f"Input: {custom_text}\n")
print("Predicted Chunk Tags (Entities):")
for entity in predictions:
    print(f"Word: {entity['word']} | Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")

Input: John works at Google in California

Predicted Chunk Tags (Entities):
Word: Google | Tag: ORG | Score: 0.9997
Word: California | Tag: LOC | Score: 0.9998


# **Task 7: Comparison**

### POS Tagging vs. Chunking

To understand how our model processes language, it is important to distinguish between Part-of-Speech (POS) Tagging and Chunking.

| Feature | POS Tagging | Chunking |
| :--- | :--- | :--- |
| **Definition** | Assigning a grammatical tag to *every* single word in a sentence (e.g., Noun, Verb, Adjective). | Grouping adjacent words that belong together into meaningful phrases (e.g., Noun Phrase, Verb Phrase). |
| **Granularity** | **Grammar-level tagging (Easy).** It looks at words individually based on their direct context. | **Phrase-level grouping (Medium).** It requires understanding higher-level structures. |
| **Example** | `[John]_PROPN` `[works]_VERB` `[at]_ADP` `[Google]_PROPN` | `[John works]_VP` `[at Google]_PP` |




# **Task 8: Report / Blog**

## 1. Key Differences Between POS Tagging and Chunking
As outlined in the comparison above, POS tagging is a foundational task. It strictly classifies the grammatical role of isolated words. Chunking (or shallow parsing) takes this a step further by using those parts of speech to map out multi-word constituents or phrases. Chunking is critical for extracting entities or complex subjects from raw text where isolated words do not tell the full story.

## 2. Challenges Faced During Implementation

While training our token classification model on the WikiANN dataset, we encountered several technical hurdles with the Hugging Face `Trainer` API:

* **The Initialization Bug:** When setting `evaluation_strategy="epoch"`, the library triggered a `RuntimeError: on_train_begin must be called before on_evaluate`. The environment attempted to evaluate baseline metrics before the training hooks successfully registered the active run state.
* **Redundant API Callbacks:** Simply overriding the `on_train_begin` state didn't clean up notebook logger hooks, leading to recurring runtime crashes at evaluation lines.
* **Saving the Ecosystem:** Leaving the tokenizer argument out of the `Trainer` prevents it from auto-bundling the vocabulary files alongside model weights during a regular `trainer.save_model()` execution.

## 3. Observations and Insights

* **The Workaround:** To preserve our goal of monitoring evaluation at the end of every epoch without fighting library bugs, we abandoned the automated `trainer.evaluate()` call. Instead, we ran `trainer.train()` for 1 epoch at a time in a manual Python loop and pulled predictions with `trainer.predict()`.
* **Dynamic Fallbacks:** We manually invoked `tokenizer.save_pretrained(epoch_save_path)` alongside the weights. This ensured that our output directory was complete and highly portable for future pipelines.
* **Results:** At the end of 3 epochs, the model hit an F1-score of roughly **0.81** with a highly balanced precision and recall. When tested on a live sentence, it confidently extracted real-world entities like **Google (ORG)** and **California (LOC)** with confidence scores exceeding **99%**.

## 4. In-Depth Observations & Conclusion

###  Important Note on Metric Selection
During evaluation, we relied on the `seqeval` framework rather than standard sequence accuracy. This is a critical distinction in token classification:
* **The Accuracy Trap:** In a dataset like WikiANN, the vast majority of tokens are non-entities labeled as `O` (Outside). A model could theoretically achieve 90%+ accuracy simply by predicting `O` for every single word, while failing entirely at extracting actual names, organizations, or locations.
* **Our Approach:** By optimizing for **Precision, Recall, and F1-score** specifically on the entity chunks rather than the background text, we ensured that our reported success rate reflects the model's genuine ability to extract entities, rather than its ability to predict filler words.

###  Architectural Insight: Multi-Task Limitations
While POS tagging and Chunking are both Token Classification tasks, a model fine-tuned on one cannot perform the other without swapping the final classification layer. This is because the output dimension of the network's final layer is strictly locked to the number of classes in the training dataset. To build a system that does both simultaneously, one would need to implement a multi-headed network sharing a single transformer backbone.

##  Conclusion
By fine-tuning DistilBERT on the WikiANN dataset, we successfully moved beyond basic word-level Part-of-Speech tagging and into phrase-level Chunking and Named Entity Recognition.

Despite encountering library state issues with the native Hugging Face evaluation callbacks, setting up a custom manual training loop allowed us to successfully extract metrics and checkpoint the weights without compromising training integrity. Reaching an F1-score of ~0.81 highlights how powerful lightweight transformer models are for dense token classification tasks, making this architecture highly viable for real-world information extraction pipelines.